In [94]:
import pandas as pd 
import numpy as np
import random

### Parte 1

1. Haz un modelo de Regresión para predecir los cargos (columna 'charges') que deberá un cliente a su seguro. (10 pts.)

2. Interpreta los resultados del modelo: (20 pts.)
- ¿Qué te dicen los coeficientes?

In [168]:
df = pd.read_csv('data/insurance.csv')

In [169]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


In [170]:
df.describe()

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [171]:
df_dummies = pd.get_dummies(df, columns=["sex","smoker","region"])
df_dummies.head()

,age,bmi,children,charges,sex_female,sex_male,smoker_no,smoker_yes,region_northeast,region_northwest,region_southeast,region_southwest
0,19,27.900,0,16884.92400,True,False,False,True,False,False,False,True
1,18,33.770,1,1725.55230,False,True,True,False,False,False,True,False
2,28,33.000,3,4449.46200,False,True,True,False,False,False,True,False
3,33,22.705,0,21984.47061,False,True,True,False,False,True,False,False
4,32,28.880,0,3866.85520,False,True,True,False,False,True,False,False


In [172]:
df_to_model = df_dummies.drop(columns="charges")

In [173]:
import statsmodels.api as sm

X = df_to_model
y = ['charges']

X = sm.add_constant(X)

# ordinary least squares
model = sm.OLS(df_dummies[y], X.astype(float))
results = model.fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:                charges   R-squared:                       0.751
Model:                            OLS   Adj. R-squared:                  0.749
Method:                 Least Squares   F-statistic:                     500.8
Date:                Wed, 30 Apr 2025   Prob (F-statistic):               0.00
Time:                        14:47:57   Log-Likelihood:                -13548.
No. Observations:                1338   AIC:                         2.711e+04
Df Residuals:                    1329   BIC:                         2.716e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
const             -296.4168    430.507  

------
### Parte 2: Optimización


### Caso: 

Administras un proyecto de software y necesitas decidir cuántas horas  a una compañía de outsourcing de dos tipos de programadores.


### Datos:
---------

**Junior programmers**

- Costo: $7 USD / hora

- Productividad: 5 unidades de eficiencia / hora, tienen retornos decrecientes conforme pasa el tiempo debido al cansancio

**Senior programmers**

- Costo: $16 USD / hora

- Productividad: 7 unidades de eficiencia / hora, tienen retornos decrecientes conforme pasa el tiempo debido al cansancio

---------

Tienes un presupuesto de **$600 USD** y necesitas asignar horas para maximizar la eficiencia. 

Nadie puede trabajar horas negativas.

### Ejercicio:

- Se te dan las primeras funciones para calcular el pago por hora a ambos tipos de programador y el pago total. 

- Completa el código para **maximizar la eficiencia** con las restricciones del caso. (30 pts.)
    - ¿Cuántas horas de junior programmer es recomendable que contrates?
    - ¿Cuántas horas de senior programmer es recomendable que contrates?
    - ¿Cuál es la máxima eficiencia que puedes alcanzar con tu presupuesto?


In [174]:
from scipy.optimize import minimize

#### PROGRAMADOR JUNIOR
def junior_programmer_hourly_pay(x):
    return 7 * x 

def junior_programmer_hourly_efficiency(x):
    return 5 * np.log(x + 2)

#### PROGRAMADOR SENIOR
def senior_programmer_hourly_pay(x):
    return 16 * x

def senior_programmer_hourly_efficiency(x):
    return 7 * np.log(2*x +2) 

#### PAGO TOTAL
def pago_total(x):
    junior_programmers, senior_programmers = x
    return (junior_programmer_hourly_pay(junior_programmers) + senior_programmer_hourly_pay(senior_programmers))

In [175]:
#### MAXIMIZAR EFICIENCIA
def eficiencia_total(x):
    junior_programmers, senior_programmers = x
    return -(junior_programmer_hourly_efficiency(junior_programmers) + senior_programmer_hourly_efficiency(senior_programmers))

total_budget = 600

# constraints
constraints = [
    {'type': 'ineq', 'fun': lambda x: total_budget - pago_total(x)},        
]

bounds = [(0,None),(0,None)]

initial_guess = [10,10]

result = minimize(eficiencia_total, initial_guess, constraints=constraints, bounds=bounds)

print(result)


 message: Optimization terminated successfully
 success: True
  status: 0
     fun: -44.912676913217595
       x: [ 3.549e+01  2.197e+01]
     nit: 11
     jac: [-1.334e-01 -3.047e-01]
    nfev: 33
    njev: 11


In [176]:
print("Horas para emplear:")
print(f"Junior Programmer: {int(result.x[0])}")
print(f"Senior Programmer: {int(result.x[1])}")
print(f"Costo total estimado: ${pago_total(result.x):,.2f}")
print(f"Eficiencia total: {int(-result.fun)}")

Horas para emplear:
Junior Programmer: 35
Senior Programmer: 21
Costo total estimado: $600.00
Eficiencia total: 44


### Parte 3


### Caso: 

Una tienda en línea quiere incrementar la venta promedio por visitante en su sitio. Se cree que cambiando el botón de 'Carrito' a 'Comprar ahora' podría animar a los clientes a comprar más. Deciden correr un A/B test. 

Por dos semanas se corrió el experimento y te entregaron los resultados.

### Datos:

- user_id
- group:
    - A -> botón normal de 'Carrito'
    - B -> botón actualizado a 'Comprar ahora'
- purchase_value: monto de compra, si no compró nada será $0
- converted: si el cliente convirtió (1), si no convirtió (0), siempre que un cliente tenga un monto de compra en $0 converted = 0

### Ejercicio:

1. Escribe la hipótesis nula y la hipótesis alterna. (10 pts.)
2. Define tu significancia estadística. (5 pts.)
3. Realiza un test estadístico y descubre si aumentó el valor de compra. (10 pts.)
4. Interpreta los resultados. (15 pts.)
    - ¿Cambiarías el botón?

In [177]:
df_ab_test = pd.read_csv('data/ab_test_data.csv')

In [180]:
df_ab_test.head()

,user_id,group,purchase_value,converted
0,1448,B,578.818631,1
1,1115,B,643.455150,1
2,1065,B,790.783597,1
3,2288,B,0.000000,0
4,1538,B,0.000000,0


In [178]:
df_ab_test.groupby('group')['purchase_value'].agg(['count', 'mean', 'std', 'median'])

,count,mean,std,median
group,,,,
A,1264,67.564120,212.368692,0.0
B,1236,74.133886,212.768703,0.0


In [179]:
# --- Análisis Original: Prueba T sobre Valor de Compra Promedio (Como antes) ---
group_a_values = df_ab_test[df_ab_test['group'] == 'A']['purchase_value']
group_b_values = df_ab_test[df_ab_test['group'] == 'B']['purchase_value']
t_stat, p_value_ttest = stats.ttest_ind(group_a_values, group_b_values, equal_var=False)
print("\n--- T-test Results (Purchase Value) ---")
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value (two-tailed): {p_value_ttest:.4f}")


--- T-test Results (Purchase Value) ---
T-statistic: -0.7726
P-value (two-tailed): 0.4398


In [183]:
# 2. Calcular conversiones y total de observaciones por grupo
summary = df_ab_test.groupby('group')['converted'].agg(['sum', 'count'])
summary.rename(columns={'sum': 'conversions', 'count': 'total_users'}, inplace=True)
print("\n--- Conversion Summary ---")
print(summary)


--- Conversion Summary ---
       conversions  total_users
group                          
A              125         1264
B              146         1236


In [184]:
# 3. Extraer los números necesarios para la prueba Z
conversions_a = summary.loc['A', 'conversions']
total_users_a = summary.loc['A', 'total_users']

conversions_b = summary.loc['B', 'conversions']
total_users_b = summary.loc['B', 'total_users']

# 4. Realizar la Prueba Z para dos proporciones
# count: Array con el número de éxitos (conversiones) en cada grupo
# nobs: Array con el número total de observaciones (usuarios) en cada grupo
count = np.array([conversions_a, conversions_b])
nobs = np.array([total_users_a, total_users_b])

z_stat, p_value_ztest = proportions_ztest(count, nobs, alternative='two-sided') # two-sided para H_a: p_A != p_B

# 5. Mostrar resultados de la Prueba Z
print("\n--- Z-test Results (Conversion Rate) ---")
print(f"Group A Conversion Rate: {conversions_a / total_users_a:.4f}")
print(f"Group B Conversion Rate: {conversions_b / total_users_b:.4f}")
print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value (two-tailed): {p_value_ztest:.4f}")

# 6. Interpretar resultados de la Prueba Z
alpha = 0.05
if p_value_ztest < alpha:
    print(f"\nP-value is less than alpha={alpha}. Reject the null hypothesis.")
    print("There is a statistically significant difference in conversion rates between Group A and Group B.")
else:
    print(f"\nP-value is greater than or equal to alpha={alpha}. Fail to reject the null hypothesis.")
    print("There is no statistically significant difference in conversion rates between Group A and Group B.")


--- Z-test Results (Conversion Rate) ---
Group A Conversion Rate: 0.0989
Group B Conversion Rate: 0.1181
Z-statistic: -1.5463
P-value (two-tailed): 0.1220

P-value is greater than or equal to alpha=0.05. Fail to reject the null hypothesis.
There is no statistically significant difference in conversion rates between Group A and Group B.
